# Theophylline case study: mechanistically centered GP applied to real data

This notebook applies the mechanistically centered Gaussian-process model
to the `theo_sd` dataset from the `nlmixr2data` R package (the classic
single-dose theophylline study of Boeckmann, Sheiner, and Beal). Twelve
adults received a single oral dose of theophylline (~3-6 mg/kg) and were
sampled densely at 11 timepoints over the next 24 hours.

The mechanistically informed prior mean is the one-compartment oral
absorption model,

$$ m_i(t) = \frac{D_i\, k_a}{V_i (k_a - k_e)} \left(e^{-k_e t} - e^{-k_a t}\right), $$

with textbook adult theophylline values $k_a = 1.5\,\text{hr}^{-1}$,
$k_e = 0.08\,\text{hr}^{-1}$ (half-life ~9 hr), and
$V_i = 0.5 \cdot W_i$ L. The GP discrepancy uses the Matern-5/2 kernel
with weakly-informative log-normal hyperparameter priors (medians
$\sigma_f = 1.5$, $\sigma_\epsilon = 0.2$, log-SD $\tau = 0.2$), matching
the amplitude/noise prior settings of the simulation study. The length scale
is fixed at 0.5 hours for this case study.

## Setup

In [ ]:
import os, sys, warnings, time
warnings.filterwarnings('ignore')
from pathlib import Path
CODE_DIR = Path(os.environ.get('PKGP_CODE_DIR', Path.cwd())).expanduser().resolve()
if not (CODE_DIR / 'scenarios.py').is_file():
    raise RuntimeError('Run from code/ or set PKGP_CODE_DIR to its absolute path.')
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
OUTPUT_DIR = Path(os.environ.get('PKGP_OUTPUT_DIR', CODE_DIR)).expanduser().resolve()
os.environ['PKGP_OUTPUT_DIR'] = str(OUTPUT_DIR)
DATA_DIR = CODE_DIR / 'data'
RES_DIR = OUTPUT_DIR / 'results'
FIG_DIR = OUTPUT_DIR / 'figures'
RES_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyreadr

import gp, methods
from models import one_cpt_oral

plt.rcParams.update({
    'font.size': 10,
    'axes.titlesize': 11,
    'axes.labelsize': 10,
    'legend.fontsize': 9,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'lines.linewidth': 1.2,
    'figure.dpi': 100,
})
os.makedirs(FIG_DIR, exist_ok=True); os.makedirs(RES_DIR, exist_ok=True)

## Load the theophylline data

In [ ]:
df = pyreadr.read_r(str(DATA_DIR / 'theo_sd.rda'))['theo_sd']
print('rows:', df.shape[0])
print('patients:', df.ID.nunique())
print(df.head(8))

## Population pharmacokinetic parameters

In [ ]:
# Textbook adult theophylline pop-typical values (calibrated to this dataset):
#   ka ~ 1.5 /hr, ke ~ 0.08 /hr (t_half ~ 9 hr), V ~ 0.5 L/kg
KA_POP = 1.5
KE_POP = 0.08
V_PER_KG = 0.5
KERNEL = 'matern52'
SIGMA_F_SCALE = 1.5
SIGMA_EPS_SCALE = 0.2
ELL_FIXED = 0.5     # length scale held fixed (hr)
PRIOR_LOG_SD = 0.2
T_END = 30.0   # the study window is 24 hr; extrapolate slightly past
print(f'ka = {KA_POP} /hr,  ke = {KE_POP} /hr  (half-life {np.log(2)/KE_POP:.1f} hr)')
print(f'V per kg = {V_PER_KG} L/kg')
print(f'kernel={KERNEL}, sigma_f_scale={SIGMA_F_SCALE}, sigma_eps_scale={SIGMA_EPS_SCALE}, ell_fixed={ELL_FIXED}, prior_log_sd={PRIOR_LOG_SD}')

## Fit mechanistic GP to each patient

In [ ]:
def fit_one_patient(patient_df, n_paths=2000, seed=42):
    rng = np.random.default_rng(seed)
    doses = patient_df[patient_df.EVID != 0]
    obs   = patient_df[patient_df.EVID == 0]
    if len(obs) == 0 or len(doses) == 0:
        return None
    WT = float(doses.WT.iloc[0])
    D  = float(doses.AMT.iloc[0])
    t_dose = float(doses.TIME.iloc[0])
    V_i = V_PER_KG * WT

    def mean_func(t):
        return one_cpt_oral(np.asarray(t, dtype=float) - t_dose,
                            KA_POP, KE_POP, V_i, D)

    t_obs = obs.TIME.values.astype(float)
    y_obs = obs.DV.values.astype(float)
    a, b = float(t_dose), max(t_dose + T_END, float(t_obs.max() + 5.0))

    res = methods.method_mech_gp(t_obs, y_obs, mean_func, a, b,
                                 n_paths=n_paths, rng=rng, kernel=KERNEL,
                                 sigma_f_scale=SIGMA_F_SCALE,
                                 sigma_eps_scale=SIGMA_EPS_SCALE,
                                 ell_fixed=ELL_FIXED,
                                 prior_log_sd=PRIOR_LOG_SD)
    res['_window']    = (a, b)
    res['_mean_func'] = mean_func
    res['_t_obs']     = t_obs
    res['_y_obs']     = y_obs
    res['_WT']        = WT
    res['_DOSE']      = D
    return res

t0 = time.time()
patient_ids = sorted(df.ID.unique())
fits = {pid: fit_one_patient(df[df.ID == pid]) for pid in patient_ids}
print(f'fitted {sum(v is not None for v in fits.values())} patients in {time.time()-t0:.1f}s')

## Summarize patient-level posteriors

In [ ]:
rows = []
for pid, r in fits.items():
    if r is None: continue
    cmax_samples = r['_paths'].max(axis=0)
    rows.append({
        'ID': pid, 'WT': r['_WT'], 'DOSE': r['_DOSE'],
        'n_obs': len(r['_t_obs']),
        'auc': r['auc'], 'auc_lo': r['auc_lo'], 'auc_hi': r['auc_hi'],
        'cmax_med': float(np.median(cmax_samples)),
        'cmax_lo':  float(np.quantile(cmax_samples, 0.025)),
        'cmax_hi':  float(np.quantile(cmax_samples, 0.975)),
    })
summary = pd.DataFrame(rows)
summary.to_csv(os.path.join(RES_DIR, 'theo_patient_summary.csv'), index=False)
summary.round(2)

## Figure: cohort-level spaghetti plot of posterior mean curves

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.5))
for pid, r in fits.items():
    if r is None: continue
    ax.plot(r['_paths_grid'], r['_mu_grid'], color='steelblue', alpha=0.5, lw=1.0)
ax.set_xlabel('time (hours)')
ax.set_ylabel('theophylline concentration (mg/L)')
ax.set_title(f'Posterior mean curves for {sum(v is not None for v in fits.values())} theophylline patients')
ax.grid(alpha=0.3, lw=0.5)
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'theo_spaghetti.pdf'), bbox_inches='tight')
plt.show()

## Figure: posterior fits for selected representative patients

In [ ]:
chosen_ids = [
    int(summary.sort_values('WT').head(1)['ID'].iloc[0]),
    int(summary.sort_values('WT', ascending=False).head(1)['ID'].iloc[0]),
    int(summary.sort_values('cmax_med').head(1)['ID'].iloc[0]),
    int(summary.sort_values('cmax_med', ascending=False).head(1)['ID'].iloc[0]),
]
chosen_ids = list(dict.fromkeys(chosen_ids))[:4]
print('Chosen patients (ID, WT, DOSE, n_obs):')
for pid in chosen_ids:
    row = summary[summary.ID == pid].iloc[0]
    print(f'  ID={pid:2d}  WT={row.WT:.1f} kg  DOSE={row.DOSE:.0f} mg  n_obs={int(row.n_obs)}')

fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=False)
for ax in axes.ravel()[len(chosen_ids):]:
    ax.set_visible(False)
for ax, pid in zip(axes.ravel(), chosen_ids):
    r = fits[pid]
    grid = r['_paths_grid']
    mu = r['_mu_grid']
    sd = np.sqrt(r['_K_grid_diag'])
    mech_mean = r['_mean_func'](grid)

    ax.fill_between(grid, mu - 1.96*sd, mu + 1.96*sd, color='#d62728', alpha=0.18,
                    label='posterior 95% band')
    ax.plot(grid, mu, color='#d62728', lw=1.6, label='posterior mean')
    ax.plot(grid, mech_mean, color='#1f77b4', lw=1.0, linestyle='--',
            label='compartment prior mean')
    ax.scatter(r['_t_obs'], r['_y_obs'], color='black', s=28, zorder=5,
               label='observed')
    ax.set_title(f"Patient {pid}  (WT={r['_WT']:.0f} kg, dose={r['_DOSE']:.0f} mg, {len(r['_t_obs'])} obs)")
    ax.set_xlabel('time (hr)'); ax.set_ylabel('mg/L')
    ax.grid(alpha=0.3, lw=0.5)
axes[0,0].legend(loc='upper right', fontsize=8)
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'theo_four_patients.pdf'), bbox_inches='tight')
plt.show()

## Figure: posterior AUC and Cmax across the cohort

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

s_sorted = summary.sort_values('auc').reset_index(drop=True)
y = np.arange(len(s_sorted))
axes[0].errorbar(s_sorted.auc, y,
                  xerr=[s_sorted.auc - s_sorted.auc_lo, s_sorted.auc_hi - s_sorted.auc],
                  fmt='o', markersize=5, color='#d62728', ecolor='#d62728',
                  elinewidth=0.9, capsize=2)
axes[0].set_yticks(y); axes[0].set_yticklabels([f'ID {int(i)}' for i in s_sorted.ID])
axes[0].set_xlabel('AUC over [t_dose, t_dose + 30 hr] (mg$\cdot$hr/L)')
axes[0].set_title('Posterior 95% AUC interval per patient')
axes[0].grid(axis='x', alpha=0.3, lw=0.5)

s_sorted2 = summary.sort_values('cmax_med').reset_index(drop=True)
y = np.arange(len(s_sorted2))
axes[1].errorbar(s_sorted2.cmax_med, y,
                  xerr=[s_sorted2.cmax_med - s_sorted2.cmax_lo,
                        s_sorted2.cmax_hi - s_sorted2.cmax_med],
                  fmt='o', markersize=5, color='#d62728', ecolor='#d62728',
                  elinewidth=0.9, capsize=2)
axes[1].set_yticks(y); axes[1].set_yticklabels([f'ID {int(i)}' for i in s_sorted2.ID])
axes[1].set_xlabel('$C_{\max}$ (mg/L)')
axes[1].set_title('Posterior 95% $C_{\max}$ interval per patient')
axes[1].grid(axis='x', alpha=0.3, lw=0.5)

fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, 'theo_cohort_intervals.pdf'), bbox_inches='tight')
plt.show()

## Cohort summary statistics

In [ ]:
print(f'AUC posterior medians:  min={summary.auc.min():.0f}, median={summary.auc.median():.0f}, max={summary.auc.max():.0f} mg*hr/L')
print(f'Cmax posterior medians: min={summary.cmax_med.min():.2f}, median={summary.cmax_med.median():.2f}, max={summary.cmax_med.max():.2f} mg/L')
auc_w = summary.auc_hi - summary.auc_lo
cmax_w = summary.cmax_hi - summary.cmax_lo
print(f'AUC 95% interval width:  median={auc_w.median():.0f}, range=[{auc_w.min():.0f}, {auc_w.max():.0f}]')
print(f'Cmax 95% interval width: median={cmax_w.median():.2f}, range=[{cmax_w.min():.2f}, {cmax_w.max():.2f}]')

## Manuscript figure: two selected patients

Fit the manuscript panels independently with a fresh seed of 42 for every patient. These use the same explicit plotting helper and parameters as the original figure workflow. The panel window extends to the later of the stated post-dose horizon or 12 hours after the last observation; for theophylline this differs from the cohort analysis above.

In [ ]:
from plotting import fit_oral_patient, plot_two_patients

panel_fits = {}
for pid in sorted(df.ID.unique()):
    p = df[df.ID == pid]
    d = p[p.EVID != 0]; o = p[p.EVID == 0]
    if len(d) == 0 or len(o) == 0:
        continue
    panel_fits[pid] = fit_oral_patient(
        o.TIME.values.astype(float), o.DV.values.astype(float),
        float(d.TIME.iloc[0]), float(d.WT.iloc[0]), float(d.AMT.iloc[0]),
        1.5, 0.08, 0.5, ell_fixed=0.5, t_end=30.0)
plot_two_patients(panel_fits, [7, 9], FIG_DIR / 'fig_theo_patients.pdf')